# Clean the UCI hospital stays

Turn the raw stay file into one analysis table. No charts. No CMS.

**Words used here**

- **Encounter:** one hospital stay. The ID is `encounter_id`.
- **Patient:** one person (`patient_nbr`). The same person can have many stays.
- **Readmit in 30 days:** the patient came back within 30 days (`readmitted` is `<30`).
- **`?`:** missing. Not a real group like "Other."
- **Leakage:** using a fact that already gives away the answer. Someone who died in the hospital cannot be readmitted. If we leave those stays in the rate, the rate looks too good.
- **Eligible stay:** a stay we can fairly score for 30-day return. Not death, not hospice, not "still a patient," not invalid gender.
- **Utilization:** visits in the year *before* this stay (inpatient, ED, outpatient).
- **ICD-9:** the old diagnosis coding system. We group the main diagnosis (`diag_1`) into broad chapters (heart, lung, endocrine, and so on).
- **LOS:** length of stay. Here that is `time_in_hospital` (1–14 days by design).

**Steps**

1. Load the stay file and the ID label file.
2. Turn `?` into missing.
3. Attach plain-language admission, discharge, and source labels.
4. Flag 30-day return and stays we should not score.
5. Band prior visits (0 / 1 / 2+) and group the main diagnosis.
6. Check IDs and save.

Raw files in `data/raw/uci/` are not edited. There is no hospital ID you can match to CMS.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "data" / "processed").is_dir():
            return path
    raise FileNotFoundError(
        "Could not find the project root. Run this notebook from the "
        "repository folder or from notebooks/."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
RAW_UCI_DIR = PROJECT_ROOT / "data" / "raw" / "uci"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)
print(RAW_UCI_DIR.exists())

## Load and inspect

One row is one stay. `readmitted` is `NO`, `>30`, or `<30`.

In [ ]:
# IDs stay text so they are not treated as numbers.
enc_raw = pd.read_csv(
    RAW_UCI_DIR / "diabetic_data.csv",
    dtype={"encounter_id": "string", "patient_nbr": "string"},
)

print(enc_raw.shape)
enc_raw.head()

In [ ]:
enc_raw["readmitted"].value_counts(dropna=False)

In [ ]:
enc_raw["gender"].value_counts(dropna=False)

In [ ]:
# Share of rows that are literally "?". Weight is almost all missing.
for col in ["race", "weight", "payer_code", "medical_specialty", "diag_1", "diag_2", "diag_3"]:
    print(f"{col}: {(enc_raw[col] == '?').mean():.1%} are '?' ")

## ID labels

`IDS_mapping.csv` is three small lookup tables stacked in one file (admission type, then discharge, then source). We split it and attach the words.

In [ ]:
def read_id_maps(path: Path) -> dict[str, pd.DataFrame]:
    """Split the stacked UCI lookup file into three name tables."""
    raw = path.read_text(encoding="utf-8").splitlines()
    chunks = {"admission_type": [], "discharge_disposition": [], "admission_source": []}
    current = None
    for line in raw:
        if line.startswith("admission_type_id"):
            current = "admission_type"
            continue
        if line.startswith("discharge_disposition_id"):
            current = "discharge_disposition"
            continue
        if line.startswith("admission_source_id"):
            current = "admission_source"
            continue
        if not line.strip(",") or current is None:
            continue
        code, _, label = line.partition(",")
        if not code.strip():
            continue
        chunks[current].append(
            {"code": int(code), "label": label.strip().strip('"').strip()}
        )
    return {k: pd.DataFrame(v) for k, v in chunks.items()}


id_maps = read_id_maps(RAW_UCI_DIR / "IDS_mapping.csv")
for name, table in id_maps.items():
    print(name, len(table))
id_maps["discharge_disposition"].head()

## Clean

Missing tokens become blank. Death and hospice stays are flagged, not deleted, so we can still describe the full file. Rates and the later model should use `eligible_for_readmit` only.

Prior-visit bands are **0 / 1 / 2+**. Weight is dropped (97% missing).

In [ ]:
# Discharge codes where the patient cannot return to the hospital.
EXPIRED_HOSPICE = {11, 13, 14, 19, 20, 21}
STILL_PATIENT = {12}


def icd9_chapter(code) -> str:
    """Broad ICD-9 chapter from the main diagnosis."""
    if pd.isna(code):
        return pd.NA
    text = str(code).strip().upper()
    if text == "" or text == "?":
        return pd.NA
    if text.startswith("E"):
        return "external_injury"
    if text.startswith("V"):
        return "supplemental"
    try:
        n = int(float(text.split(".")[0]))
    except ValueError:
        return pd.NA
    if n <= 139:
        return "infectious"
    if n <= 239:
        return "neoplasm"
    if n <= 279:
        return "endocrine"  # includes diabetes (250)
    if n <= 289:
        return "blood"
    if n <= 319:
        return "mental"
    if n <= 389:
        return "nervous"
    if n <= 459:
        return "circulatory"
    if n <= 519:
        return "respiratory"
    if n <= 579:
        return "digestive"
    if n <= 629:
        return "genitourinary"
    if n <= 679:
        return "pregnancy"
    if n <= 709:
        return "skin"
    if n <= 739:
        return "musculoskeletal"
    if n <= 759:
        return "congenital"
    if n <= 779:
        return "perinatal"
    if n <= 799:
        return "symptoms"
    if n <= 999:
        return "injury"
    return pd.NA


def visit_band(n: int) -> str:
    """Lock prior-visit groups at 0 / 1 / 2+ after looking at the counts."""
    if n <= 0:
        return "0"
    if n == 1:
        return "1"
    return "2+"


enc = enc_raw.copy()

# "?" is missing, not a category named question mark.
missing_object_cols = [
    "race", "weight", "payer_code", "medical_specialty",
    "diag_1", "diag_2", "diag_3",
]
for col in missing_object_cols:
    enc[col] = enc[col].replace("?", pd.NA)

enc["gender"] = enc["gender"].replace({"Unknown/Invalid": pd.NA})

# Lab fields: blank / None means the test was not done.
for col in ["max_glu_serum", "A1Cresult"]:
    enc[col] = enc[col].replace({"None": pd.NA, "none": pd.NA})

# Attach the words from the ID file.
enc = enc.merge(
    id_maps["admission_type"].rename(columns={"code": "admission_type_id", "label": "admission_type"}),
    on="admission_type_id",
    how="left",
    validate="many_to_one",
)
enc = enc.merge(
    id_maps["discharge_disposition"].rename(
        columns={"code": "discharge_disposition_id", "label": "discharge_disposition"}
    ),
    on="discharge_disposition_id",
    how="left",
    validate="many_to_one",
)
enc = enc.merge(
    id_maps["admission_source"].rename(
        columns={"code": "admission_source_id", "label": "admission_source"}
    ),
    on="admission_source_id",
    how="left",
    validate="many_to_one",
)

# Clean leftover "NULL" / "Not Available" labels the same way as missing IDs.
blank_labels = {"NULL", "Not Available", "Not Mapped", "Unknown/Invalid"}
for col in ["admission_type", "discharge_disposition", "admission_source"]:
    enc[col] = enc[col].mask(enc[col].isin(blank_labels))

enc["readmit_30"] = enc["readmitted"].eq("<30").astype(int)

enc["cannot_return"] = enc["discharge_disposition_id"].isin(EXPIRED_HOSPICE | STILL_PATIENT)
enc["eligible_for_readmit"] = (~enc["cannot_return"]) & enc["gender"].notna()

enc["inpatient_band"] = enc["number_inpatient"].map(visit_band)
enc["emergency_band"] = enc["number_emergency"].map(visit_band)
enc["outpatient_band"] = enc["number_outpatient"].map(visit_band)
# Acute use before this stay: inpatient + ED, same 0 / 1 / 2+ cuts.
enc["prior_acute_band"] = (enc["number_inpatient"] + enc["number_emergency"]).map(visit_band)

enc["diag_1_group"] = enc["diag_1"].map(icd9_chapter)

# One spelling per label.
enc["race"] = enc["race"].replace({"AfricanAmerican": "African American"})
enc["change"] = enc["change"].replace({"Ch": "Yes"})
enc["max_glu_serum"] = enc["max_glu_serum"].replace({"Norm": "Normal"})
enc["A1Cresult"] = enc["A1Cresult"].replace({"Norm": "Normal"})
enc["payer_code"] = enc["payer_code"].astype("string")
enc["admission_source"] = enc["admission_source"].replace(
    {"Transfer from critial access hospital": "Transfer from critical access hospital"}
)

# Weight is almost empty. Keep it out of the mart.
enc = enc.drop(columns=["weight"])

print(enc.shape)
print("eligible", int(enc["eligible_for_readmit"].sum()))
print("not eligible", int((~enc["eligible_for_readmit"]).sum()))
enc.head()

## Checks

In [ ]:
assert enc["encounter_id"].nunique() == len(enc)
assert enc["encounter_id"].duplicated().sum() == 0
assert enc["readmit_30"].isin([0, 1]).all()
assert enc.loc[enc["readmitted"].eq("<30"), "readmit_30"].eq(1).all()
assert (enc["race"] == "?").sum() == 0
assert enc["admission_type"].notna().any()
assert enc["discharge_disposition"].notna().any()

n_expired = enc["discharge_disposition_id"].isin(EXPIRED_HOSPICE | STILL_PATIENT).sum()
n_bad_gender = enc["gender"].isna().sum()
print("stays", len(enc))
print("people", enc["patient_nbr"].nunique())
print("30-day returns", int(enc["readmit_30"].sum()))
print("cannot return (death/hospice/still in)", int(n_expired))
print("invalid gender", int(n_bad_gender))
print("eligible stays", int(enc["eligible_for_readmit"].sum()))
print(enc["prior_acute_band"].value_counts().sort_index().to_string())
print(enc["diag_1_group"].value_counts(dropna=False).head(10).to_string())

## Save

One file: `data/processed/uci_encounter_mart.csv`. One row is one stay. Filter to `eligible_for_readmit` before you quote a 30-day rate or fit the model.

In [ ]:
out = PROCESSED_DATA_DIR / "uci_encounter_mart.csv"
enc.to_csv(out, index=False)
print(out.name, enc.shape)